In [13]:
from DeviceClasses.E5063A_VNA_Class import VNA
from DeviceClasses.RS_FSW_new import FSW
from DeviceClasses.SMA100B import Signal_Generator
from DeviceClasses.Utilities.Fitting import fit_S21_peak, S21_eq, linear_voltage_to_dB, dBm_to_watts, watts_to_dBm, RMS_to_dBm, dBm_to_RMS, RMS_squared_to_dBm
from DeviceClasses.Utilities.data_management import read_file
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import os
%matplotlib qt



In [14]:


base_path = r"D:\Bulk_He_2_1\data\attenuation_data\2025_01_16_circuit_1_SG_to_blue_input_and_ffm\\"
S21_data_path = base_path


In [15]:
#organisation of dictionary
# SA_data = {"10[dBm]":{
#     "average_21":{
#         "freq"          :{},
#         "time"          :{},
#         "IQ_voltage_mag":{},
#     }
# }}
S21_data = {}

#load in file
for file in os.listdir(S21_data_path):
    path =   S21_data_path + file
    print(f"file:{file}")

    data_pandas_temp, meta_data_temp = read_file(path)
    print(f"meta_data:{meta_data_temp.keys()}")
    Result  = meta_data_temp["Result"]
    time_start = meta_data_temp["time_start[s]"]


    if not Result in S21_data.keys():
        S21_data[Result] = {}
    S21_data[Result][time_start] = {"data":data_pandas_temp,
                                    "meta_data":meta_data_temp}
    
                    

file:2025_01_16_17_16_15_870963_S21check_0_S21_0_.tsv
meta_data:dict_keys(['time_start[s]', 'delta_T[s]', 'Result', 'TraceName', 'freq_start', 'freq_stop', 'number_of_points', 'probe_power', 'band_width', 'calibration', 'load_state', 'sense_average_state', 'sense_average_count_factor', 'sense_average_point_count_factor', 'sense_sweep_type_format', 'reset', '_file_name', 'file_location', 'extra_meta', 'device_cal', 'sweep_time'])
file:2025_01_16_17_17_04_654055_S21check_1_S21_0_.tsv
meta_data:dict_keys(['time_start[s]', 'delta_T[s]', 'Result', 'TraceName', 'freq_start', 'freq_stop', 'number_of_points', 'probe_power', 'band_width', 'calibration', 'load_state', 'sense_average_state', 'sense_average_count_factor', 'sense_average_point_count_factor', 'sense_sweep_type_format', 'reset', '_file_name', 'file_location', 'extra_meta', 'device_cal', 'sweep_time'])
file:2025_01_16_17_17_53_454516_S21check_2_S21_0_.tsv
meta_data:dict_keys(['time_start[s]', 'delta_T[s]', 'Result', 'TraceName', 'freq

In [16]:
def save_data_3(data, file_name, file_location = "", headers = None, meta_data = {}, delimiter = "\t"):
    
    data_shape = np.shape(data)
    print(f"datashape:{data_shape}")
    with open(file_location+file_name, "w") as file1:
        for key, value in meta_data.items():
            file1.write(f"#\t{key}\t{value}\n")
        if headers != None:
            file1.write(headers+"\n")
        
        try:
            for  i in range(data_shape[1]):
                for l in range(data_shape[0]):
                    file1.write(f"{data[l,i]}\t")
                file1.write("\n")
        except IndexError as e:
            print(f"index error:\n{e}\ndata:\n{data}")
            for  i in range(data_shape[0]):
                file1.write(f"{data[i]}\t")
                file1.write("\n")


#https://knowledge.ni.com/KnowledgeArticleDetails?id=kA03q000000ow2XCAQ&l=en-GBs

def Vpp_to_Vrms(V_pp):
    Vrms = V_pp/(2*np.sqrt(2))
    return Vrms

def Vp_to_Vrms(V_pp):
    Vrms = V_pp/(np.sqrt(2))
    return Vrms

def Vrms_to_Vpp(Vrms):
    V_pp = Vrms*(2*np.sqrt(2))
    return V_pp

def Vrms_to_Vp(Vrms):
    V_p = Vrms*(np.sqrt(2))
    return V_p

def Vrms_to_watts(Vrms, Z_0 = 50):
    return (Vrms**2)/Z_0

def watts_to_dBw(watts):
    return 10*np.log10(watts)

def RMS_to_dBw(Vrms, Z_0 = 50):
    watts = Vrms_to_watts(Vrms, Z_0)
    return watts_to_dBw(watts)

def RMS_to_dBm(Vrms, Z_0 = 50):
    return RMS_to_dBw(Vrms, Z_0) + 30

def dBw_to_watts(dBw):
    return 10**(dBw/10)

def dBm_to_dBw(dBm):
    return dBm - 30

def dBm_to_RMS(dBm, Z_0 = 50):
    dBw = dBm_to_dBw(dBm)
    watts = dBw_to_watts(dBw)
    Vrms = np.sqrt(watts * Z_0)
    return Vrms

def watts_to_dBm(watts):
    dBw = watts_to_dBw(watts)
    return dBw + 30

def dBm_to_watts(dBm):
    dBw = dBm_to_dBw(dBm)
    watts = dBw_to_watts(dBw)
    return watts

def Vrms_squared_to_watts(Vrms_squared,Z_0 = 50):
    watts = Vrms_squared/Z_0
    return watts





# def RMS_to_dBm(V, Z_0 = 50):
#     return 20*np.log10((V)/(np.sqrt(Z_0))) +30

# def RMS_squared_to_dBm(V_square, Z_0 = 50):
#     return 10*np.log10((V_square)/(0.001*Z_0))

# def dBm_to_RMS(dBm, Z_0 = 50):
#     return np.sqrt((Z_0*0.001)*10**(dBm/10))

In [17]:
print(f"data:{S21_data.keys()}")
#SA_data["10[dBm]"].pop("average_0")


data:dict_keys(['S21'])


In [20]:
plt.close()

fig,ax = plt.subplots(1)
ax.set(title = "S-parameters - Attenuation", xlabel = "frequency[Hz]", ylabel = "power_ratio[dB]")
fig.tight_layout()
print(f"data:{S21_data.keys()}")


for result, result_dict in S21_data.items():
    for time_start, time_start_dict in result_dict.items():
        data_pandas_temp = time_start_dict["data"]
        meta_data = time_start_dict["meta_data"]

        mag = data_pandas_temp["sweep_number_mag[dB]:1"]
        freq = data_pandas_temp["freq[Hz]"]
        ax.plot(freq, mag, label = f"{result}, {time_start}")


ax.legend()


fig.show()

data:dict_keys(['S21'])


In [7]:
31250.0/0.016016

1951173.8261738264

Traceback (most recent call last):
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\backends\backend_qt.py", line 459, in _draw_idle
    self.draw()
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\backends\backend_agg.py", line 388, in draw
    self.figure.draw(self.renderer)
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\artist.py", line 95, in draw_wrapper
    result = draw(artist, renderer, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\artist.py", line 72, in draw_wrapper
    return draw(artist, renderer)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\figure.py", line 3154, in draw
    mimage._draw_list_compositing_images(
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\image.py", line 132, in _draw_list_compositing_images
    a.draw(renderer)
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\artist.py", line 72, in d

Traceback (most recent call last):
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\backends\backend_qt.py", line 459, in _draw_idle
    self.draw()
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\backends\backend_agg.py", line 388, in draw
    self.figure.draw(self.renderer)
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\artist.py", line 95, in draw_wrapper
    result = draw(artist, renderer, *args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\artist.py", line 72, in draw_wrapper
    return draw(artist, renderer)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\figure.py", line 3154, in draw
    mimage._draw_list_compositing_images(
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\image.py", line 132, in _draw_list_compositing_images
    a.draw(renderer)
  File "c:\python\anaconda3\Lib\site-packages\matplotlib\artist.py", line 72, in d